# Lab 4.12 &mdash; Capstone: Adapt a Pre-trained Model to a New Task

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 45 min &nbsp;|&nbsp; **Day 2 &middot; Module 4 &mdash; Pre-trained Models & Fine-tuning**

### What you'll do
- Apply the whole Module 4 pipeline to a NEW task (topic detection)
- Build features, train a head, and hold out a validation set
- Evaluate and predict on unseen text

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

**Reference:** [Module 4 slides &mdash; Fine-tuning end-to-end](../../presentation/day2-module4-pretrained-models-and-fine-tuning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 4 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-04-12"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
Capstone: the full transfer-learning workflow on a **new** task &mdash; classify a sentence as
**sports** vs **tech**. Same recipe as sentiment: frozen features + trainable head + held-out
evaluation. This is exactly how you would adapt a pre-trained model to *your* problem.

In [ ]:
# DEMO -- a new tiny task: sports (0) vs tech (1). Keywords recur across rows
# (team/goal/game/player... vs app/software/data/chip...) so the model generalises.
TASK = [
    ("the team scored a goal to win the game", 0),
    ("our team won the match with a late goal", 0),
    ("the player scored twice in the game", 0),
    ("a great goal helped the team win the match", 0),
    ("the coach and team celebrated the win", 0),
    ("the player passed the ball and scored a goal", 0),
    ("the team lost the game by one goal", 0),
    ("the new app runs on a fast chip", 1),
    ("the software update improved the app", 1),
    ("the app stores its data in the cloud", 1),
    ("a new chip makes the computer software faster", 1),
    ("the cloud server runs the data software", 1),
    ("the app uses data and a fast chip", 1),
    ("the computer software had a data bug", 1),
]
print("examples:", len(TASK))

## Your Turn
Split, fit a frozen extractor + head, evaluate, and predict on new sentences.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X = [t for t, y in TASK]; Y = [y for t, y in TASK]
Xtr_t, Xval_t, ytr, yval = train_test_split(X, Y, test_size=0.3, random_state=0, stratify=Y)
vec = TfidfVectorizer()
head = LogisticRegression(max_iter=1000)

def fit_and_eval():
    Xtr = ___   # TODO: vec.fit_transform on the train text
    head.fit(Xtr, ytr)   # train the head
    return accuracy_score(yval, head.predict(___))   # TODO: vectorise val text

def classify(sentence):
    return int(head.predict(vec.transform([sentence]))[0])

try:
    print('val accuracy:', round(fit_and_eval(), 3))
    print('"the goalkeeper made a great save" ->', classify('the player scored a goal for the team'))
    print('"the gpu accelerates the neural network" ->', classify('the app stores data on the cloud server'))
except Exception as e: print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("end-to-end pipeline returns an accuracy", lambda: isinstance(fit_and_eval(), float))
expect_true("val accuracy >= 0.8 on the new task", lambda: fit_and_eval() >= 0.8)
expect_true("classifies a sports sentence as 0", lambda: (fit_and_eval(), classify('the player scored a goal for the team'))[1] == 0)
expect_true("classifies a tech sentence as 1", lambda: (fit_and_eval(), classify('the app stores data on the cloud server'))[1] == 1)

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; the real thing with Hugging Face (not graded)
Adapt a REAL pretrained model to this task with zero-shot classification (no training). Safe to skip &mdash; it needs `pip install transformers torch` and a one-time model
download. If `transformers` is not installed (or offline), the cell prints a note and moves on.
**Real BERT fine-tuning is the managed-sandbox target; the graded steps above run offline so the
lab always works.**

In [ ]:
try:
    from transformers import pipeline
    zs = pipeline("zero-shot-classification")
    out = zs("the gpu accelerates the neural network", candidate_labels=["sports", "tech"])
    print(out["labels"][0], "(", round(out["scores"][0], 3), ")")
    print("Pre-trained models can even classify with NO task training -- zero-shot.")
except Exception as e:
    print("Optional real-model cell skipped -- your trained head above already solves the task.")
    print("  reason:", type(e).__name__, "--", e)

---
### Key takeaway
You adapted the pre-trained pipeline to a brand-new task end-to-end. That is Module 4 in one move: stand on a pre-trained model, add a small head, evaluate honestly, ship. Next: Day 3 &mdash; agents.

[&#8592; All Module 4 labs](./index.html) &nbsp;&middot;&nbsp; [Module 4 slides](../../presentation/day2-module4-pretrained-models-and-fine-tuning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>